<a href="https://colab.research.google.com/github/Ammaryasir4673/My_Python_Projects/blob/main/Machine_Learning/1st_pytorch_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pymatgen


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.7/829.7 kB 19.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 97.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 979.1/979.1 kB 43.5 MB/s  0:00:00
  Created wheel for bibtexparser: filename=bibtexparser-1.4.4-py3-none-any.whl size=43686 sha256=3a68d7c09a621cd9536c383ed8539414ec2d8bc1ccdb5e21d7e7ae24a0c9d984
  Stored in directory: /root/.cache/pip/wheels/43/ca/d8/0d315657fcf8ff8694a2f6701589bd1eaa602a85b7a7810063
Successfully built bibtexparser
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [pymatgen]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This 

In [ ]:
#import libraries

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from pymatgen.core import Lattice, Structure

In [ ]:
#create a crystal structure

In [ ]:
#We provide only one atomic position, and pymatgen generates the remaining positions automatically.

In [ ]:
lattice = Lattice.cubic(5.431)

structure = Structure.from_spacegroup(
    "Fd-3m", #space group name
    lattice,
    ["Si"],
    [[0, 0, 0]] #This specifies the fractional coordinates of the initial atom.

                #Fractional coordinates describe an atomic position relative to the lattice vectors.
)

print(structure)
print("Number of atoms:", len(structure))

Full Formula (Si8)
Reduced Formula: Si
abc   :   5.431000   5.431000   5.431000
angles:  90.000000  90.000000  90.000000
pbc   :       True       True       True
Sites (8)
  #  SP       a     b     c
---  ----  ----  ----  ----
  0  Si    0.5   0     0.5
  1  Si    0.25  0.25  0.25
  2  Si    0     0     0
  3  Si    0.25  0.75  0.75
  4  Si    0.75  0.25  0.75
  5  Si    0     0.5   0.5
  6  Si    0.5   0.5   0
  7  Si    0.75  0.75  0.25
Number of atoms: 8


In [ ]:
#Convert atoms into graph nodes

In [ ]:
x = []

for site in structure:

    atomic_number = site.specie.Z
    atomic_mass = float(site.specie.atomic_mass)

    x.append([atomic_number, atomic_mass])

x = torch.tensor(x, dtype=torch.float)

print(x)
print("Node feature shape:", x.shape)

tensor([[14.0000, 28.0855],
        [14.0000, 28.0855],
        [14.0000, 28.0855],
        [14.0000, 28.0855],
        [14.0000, 28.0855],
        [14.0000, 28.0855],
        [14.0000, 28.0855],
        [14.0000, 28.0855]])
Node feature shape: torch.Size([8, 2])


In [ ]:
#Connect neighboring atoms (edges)

In [ ]:
edge_list = []

for i, neighbors in enumerate(
    structure.get_all_neighbors(r=3.0)
):

    for neighbor in neighbors:

        j = neighbor.index

        edge_list.append([i, j])

edge_index = torch.tensor(
    edge_list,
    dtype=torch.long
).T

print(edge_index)
print("Edge index shape:", edge_index.shape)

tensor([[0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 4, 4, 4, 4, 5, 5, 5, 5,
         6, 6, 6, 6, 7, 7, 7, 7],
        [3, 1, 7, 4, 6, 5, 2, 0, 7, 4, 3, 1, 5, 6, 0, 2, 0, 6, 5, 2, 4, 7, 1, 3,
         1, 3, 4, 7, 6, 0, 5, 2]])
Edge index shape: torch.Size([2, 32])


In [ ]:
#above show each atom connected with 4 other atoms so their are 32 total connections

In [ ]:
#- First row: Starting atom (source node).
#- Second row: Neighboring atom (destination node).

In [ ]:
#Build a GNN from scratch

In [ ]:
num_nodes = x.shape[0]

A = torch.zeros(num_nodes, num_nodes)

for i, j in edge_index.T:
    A[i, j] = 1

print(A)

tensor([[0., 1., 0., 1., 1., 0., 0., 1.],
        [1., 0., 1., 0., 0., 1., 1., 0.],
        [0., 1., 0., 1., 1., 0., 0., 1.],
        [1., 0., 1., 0., 0., 1., 1., 0.],
        [1., 0., 1., 0., 0., 1., 1., 0.],
        [0., 1., 0., 1., 1., 0., 0., 1.],
        [0., 1., 0., 1., 1., 0., 0., 1.],
        [1., 0., 1., 0., 0., 1., 1., 0.]])


In [ ]:
#Normalize the adjacency matrix
#We add self-connections so that each atom can retain its own information.

In [ ]:
I = torch.eye(num_nodes)

A_hat = A + I

degree = A_hat.sum(dim=1) #Degree means the number of connections a node has.

D_inv = torch.diag(1 / degree)

A_norm = D_inv @ A_hat #multiplication

print(A_norm)

tensor([[0.2000, 0.2000, 0.0000, 0.2000, 0.2000, 0.0000, 0.0000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.2000, 0.2000, 0.0000],
        [0.0000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.2000],
        [0.2000, 0.0000, 0.2000, 0.2000, 0.0000, 0.2000, 0.2000, 0.0000],
        [0.2000, 0.0000, 0.2000, 0.0000, 0.2000, 0.2000, 0.2000, 0.0000],
        [0.0000, 0.2000, 0.0000, 0.2000, 0.2000, 0.2000, 0.0000, 0.2000],
        [0.0000, 0.2000, 0.0000, 0.2000, 0.2000, 0.0000, 0.2000, 0.2000],
        [0.2000, 0.0000, 0.2000, 0.0000, 0.0000, 0.2000, 0.2000, 0.2000]])


In [ ]:
# 0.2 shows the weights of atom. e.g atom has 4 connections and 1 itself so total 5 degree.
# so each atom constibutes 0.2. b/c 1/5=0.2

In [ ]:
#above shows self atom and negibour atom informatins added.

In [ ]:
#create GNN model

In [ ]:
class GNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.layer1 = nn.Linear(2, 8)

        self.layer2 = nn.Linear(8, 4)

        self.output_layer = nn.Linear(4, 1)


    def forward(self, x, A_norm):

        # First message passing
        x = A_norm @ x

        x = self.layer1(x)

        x = F.relu(x)


        # Second message passing
        x = A_norm @ x

        x = self.layer2(x)

        x = F.relu(x)


        # Graph pooling
        x = x.mean(dim=0)


        # Final prediction
        x = self.output_layer(x)

        return x

In [ ]:
#run model

In [ ]:
model = GNN()

output = model(x, A_norm)

print(output)

print("Output shape:", output.shape)

tensor([-1.1211], grad_fn=<ViewBackward0>)
Output shape: torch.Size([1])


In [ ]:
#get one representation for whole atoms

In [ ]:
graph_embedding = output.mean(dim=0)

print(graph_embedding)
print(graph_embedding.shape)

tensor(-1.1211, grad_fn=<MeanBackward1>)
torch.Size([])


In [ ]:
#Create model, loss function and optimizer

In [ ]:
model = GNN()

target = torch.tensor([1.12], dtype=torch.float)

loss_fn = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
#Training loop

In [ ]:
epochs = 5

for epoch in range(epochs):

    optimizer.zero_grad()

    prediction = model(x, A_norm)

    loss = loss_fn(prediction, target)

    loss.backward()

    optimizer.step()

    print(
        f"Epoch [{epoch + 1}/{epochs}], "
        f"Prediction: {prediction.item():.4f}, "
        f"Loss: {loss.item():.4f}"
    )

Epoch [1/5], Prediction: -0.4964, Loss: 2.6127
Epoch [2/5], Prediction: -0.4954, Loss: 2.6094
Epoch [3/5], Prediction: -0.4944, Loss: 2.6062
Epoch [4/5], Prediction: -0.4934, Loss: 2.6030
Epoch [5/5], Prediction: -0.4924, Loss: 2.5997


In [ ]:
#evaluate and final output

In [ ]:
model.eval()

with torch.no_grad():

    prediction = model(x, A_norm)

print("Predicted band gap:", prediction.item(), "eV")

Predicted band gap: -0.4913698136806488 eV


In [ ]:
# we can use built in python libraries for built a model. it will give x_new. we do not need to scratch from crystal.
#simply we  do not need to manually construct A, A_norm, or calculate A_norm @ x